# Banking Transactions Data Pipeline (PySpark, medallion layers)

This notebook is the distributed version of the pandas pipeline. It follows the **medallion** pattern:

- **Bronze:** the raw file is landed as Parquet, unchanged, with load metadata. This is the *E* and *L* of ELT.
- **Silver:** data is cleaned, validated and split into clean rows and quarantine rows with reason codes. This is the *T*, done inside the lakehouse.
- **Gold:** business summaries built from silver for reporting.

Every code line has a comment. The notebook runs locally with `local[*]`. On a Databricks or Spark cluster you remove the `.master(...)` line and submit the same code.


In [ ]:
import os  # load os to build folder paths
import shutil  # load shutil to delete old results before a fresh run
from pyspark.sql import SparkSession  # the entry point for all Spark work
from pyspark.sql import functions as F  # column functions such as when, col, lit and sum
from pyspark.sql import Window  # window functions used to rank duplicate rows

In [ ]:
spark = (
    SparkSession.builder  # start building a Spark session
    .appName('bank_data_pipeline')  # give the job a name so it is easy to find in the Spark UI
    .master('local[*]')  # run on this machine using all CPU cores; remove this line on a cluster
    .config('spark.sql.shuffle.partitions', '8')  # use few shuffle partitions because the data is small
    .getOrCreate()  # create the session, or reuse the one already running
)
spark.sparkContext.setLogLevel('WARN')  # show only warnings and errors so the output stays readable

## 0. Paths
The `lakehouse/` folder stands in for cloud storage. In production these would be paths such as `abfss://` or `s3://`.

In [ ]:
INPUT_FILE = 'bank_transactions.csv'  # the raw landing file
BASE = 'lakehouse'  # root folder that stands in for cloud storage
BRONZE = os.path.join(BASE, 'bronze')  # raw data plus load metadata
SILVER = os.path.join(BASE, 'silver')  # cleaned and validated data
GOLD = os.path.join(BASE, 'gold')  # business-ready summaries
OUTPUT_DIR = 'output_spark'  # folder for CSV extracts you can open in Excel
shutil.rmtree(BASE, ignore_errors=True)  # delete the previous lakehouse so every run starts clean
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)  # delete the previous CSV extracts
os.makedirs(OUTPUT_DIR, exist_ok=True)  # recreate the output folder

## 1. Bronze: land the raw data
Read every column as text so nothing is converted silently, then keep the rows exactly as they arrived.

In [ ]:
raw_df = (
    spark.read  # start a reader on the file source
    .option('header', True)  # the first line contains the column names
    .csv(INPUT_FILE)  # read the file; without a schema option every column is read as text
)
bronze_df = (
    raw_df  # start from the raw rows
    .withColumn('_source_file', F.col('_metadata.file_path'))  # record which file each row came from
    .withColumn('_ingested_at', F.current_timestamp())  # record when this batch was loaded
)
bronze_df.write.mode('overwrite').parquet(os.path.join(BRONZE, 'transactions'))  # save as Parquet, a compressed columnar format
print('Bronze rows loaded:', spark.read.parquet(os.path.join(BRONZE, 'transactions')).count())  # confirm the number of rows written

## 2. Silver: clean, type, de-duplicate
Fix text, convert types with `try_` functions so bad values become null instead of crashing, and remove exact copies.

In [ ]:
typed_df = (
    bronze_df  # start from the bronze layer
    .withColumn('transaction_id', F.trim('transaction_id'))  # remove stray spaces from IDs
    .withColumn('account_id', F.nullif(F.trim('account_id'), F.lit('')))  # turn a blank account ID into null
    .withColumn('transaction_type', F.upper(F.trim('transaction_type')))  # turn 'debit ' into 'DEBIT'
    .withColumn('currency', F.upper(F.trim('currency')))  # turn 'mur' into 'MUR'
    .withColumn('merchant_category', F.coalesce(F.nullif(F.trim('merchant_category'), F.lit('')), F.lit('UNKNOWN')))  # fill an empty category with 'UNKNOWN'
    .withColumn('status', F.upper(F.trim('status')))  # standardise status text to upper case
    .withColumn('transaction_date', F.try_to_date(F.trim('transaction_date'), 'yyyy-MM-dd'))  # parse the date; impossible dates such as 2026-02-30 become null
    .withColumn('amount', F.expr('try_cast(trim(amount) as decimal(18,2))'))  # convert to a decimal number; empty or invalid text becomes null
    .withColumn('balance_after', F.expr('try_cast(trim(balance_after) as decimal(18,2))'))  # convert the balance the same way
)
business_cols = [c for c in typed_df.columns if c not in ('transaction_id', '_source_file', '_ingested_at')]  # the fields that describe the money movement
dedupe_window = Window.partitionBy(*business_cols).orderBy('transaction_id')  # group identical transactions and order each group by ID
deduped_df = (
    typed_df  # start from the typed rows
    .withColumn('_rn', F.row_number().over(dedupe_window))  # number the copies in each group; the first copy gets 1
    .filter(F.col('_rn') == 1)  # keep only the first copy of each transaction
    .drop('_rn')  # remove the helper column
)
print('Typed rows:', typed_df.count(), '| after removing duplicate copies:', deduped_df.count())  # show the effect of de-duplication

## 3. Validate with reason codes
Each failed rule adds a code. Rows with any code go to quarantine, which keeps the reason for review. Clean rows are the rest.

In [ ]:
reason_df = (
    deduped_df  # start from the de-duplicated rows
    .withColumn('reason_codes', F.concat_ws(',',  # join every failed rule into one comma-separated text value
        F.when(F.col('account_id').isNull(), F.lit('MISSING_ACCOUNT')),  # no account, so it cannot be linked to a customer
        F.when(F.col('amount').isNull(), F.lit('MISSING_AMOUNT')),  # no money value, and we never guess one
        F.when(F.col('transaction_date').isNull(), F.lit('INVALID_DATE')),  # date missing or impossible
        F.when(F.col('amount') < 0, F.lit('NEGATIVE_AMOUNT')),  # negative amount, so the sign convention is unclear
        F.when(F.col('status') != 'SUCCESS', F.lit('FAILED_STATUS')),  # failed transactions did not move money
    ))
)
quarantine_df = reason_df.filter(F.col('reason_codes') != '')  # rows with at least one failed rule, with their reason codes
clean_df = reason_df.filter(F.col('reason_codes') == '').drop('reason_codes')  # rows that passed every rule, without the empty code column
print('Clean rows:', clean_df.count(), '| Quarantined rows:', quarantine_df.count())  # show how many rows went to each group

## 4. Transform and flag
Add the signed amount, month, hour, large-transaction flag, and a currency-mismatch flag for each account (problem 13).

In [ ]:
acct_currency = (
    clean_df  # start from clean rows
    .groupBy('account_id')  # one row per account
    .agg(F.count_distinct('currency').alias('currency_count'))  # count how many different currencies each account uses
)
silver_df = (
    clean_df  # start from clean rows
    .join(acct_currency, on='account_id', how='left')  # attach the currency count for each account
    .withColumn('currency_mismatch', F.col('currency_count') > 1)  # True when the account has more than one currency, so it needs review
    .drop('currency_count')  # remove the helper column
    .withColumn('signed_amount', F.when(F.col('transaction_type') == 'CREDIT', F.col('amount')).otherwise(-F.col('amount')))  # positive for money in, negative for money out
    .withColumn('txn_month', F.date_format('transaction_date', 'yyyy-MM'))  # month label such as 2026-09, used for partitions and reports
    .withColumn('txn_hour', F.hour(F.try_to_timestamp(F.col('transaction_time'), F.lit('HH:mm:ss'))))  # the hour of the day from the time
    .withColumn('is_large_txn', F.col('amount') >= 10000)  # flag amounts of 10,000 or more for AML review
)
silver_df.write.mode('overwrite').partitionBy('txn_month').parquet(os.path.join(SILVER, 'transactions'))  # save clean rows as Parquet, one folder per month
quarantine_df.write.mode('overwrite').parquet(os.path.join(SILVER, 'quarantine'))  # save the rejected rows with their reason codes

## 5. Data quality checks
The checks confirm that every raw row is accounted for, and that the silver layer obeys its rules. A failed check stops the job.

In [ ]:
bronze_rows = spark.read.parquet(os.path.join(BRONZE, 'transactions')).count()  # rows loaded from the raw file
silver_rows = silver_df.count()  # rows in the clean layer
quarantine_rows = quarantine_df.count()  # rows in the quarantine layer
duplicates_removed = typed_df.count() - deduped_df.count()  # exact copies removed during de-duplication
assert bronze_rows == silver_rows + quarantine_rows + duplicates_removed, 'Row counts do not reconcile'  # every raw row must be accounted for
assert silver_df.select('transaction_id').distinct().count() == silver_rows, 'Duplicate IDs in silver'  # IDs must be unique in the clean layer
assert silver_df.filter(F.col('amount') <= 0).count() == 0, 'Zero or negative amounts in silver'  # clean amounts must be positive
assert silver_df.filter(F.col('account_id').isNull()).count() == 0, 'Missing account IDs in silver'  # every clean row must link to an account
print('All data quality checks passed')  # confirm success
print('Reconciliation: raw', bronze_rows, '= clean', silver_rows, '+ quarantine', quarantine_rows, '+ duplicates removed', duplicates_removed)  # show the row accounting
quarantine_df.select(F.explode(F.split('reason_codes', ',')).alias('reason')).groupBy('reason').count().show()  # count how often each reason code occurred

## 6. Gold: business summaries
Gold reads from silver, as a separate downstream job would. Currencies are kept separate so that MUR and USD are never added together.

In [ ]:
silver = spark.read.parquet(os.path.join(SILVER, 'transactions'))  # read clean data back from the silver layer
monthly_account = (
    silver  # start from clean rows
    .groupBy('account_id', 'currency', 'txn_month')  # one row per account, currency and month
    .agg(
        F.sum('signed_amount').alias('net_flow'),  # total money in minus money out
        F.count('transaction_id').alias('txn_count'),  # number of transactions
    )
    .orderBy('account_id', 'currency', 'txn_month')  # sort the output for easy reading
)
channel_summary = (
    silver  # start from clean rows
    .groupBy('channel')  # one row per channel
    .agg(
        F.count('transaction_id').alias('txn_count'),  # number of transactions per channel
        F.sum('amount').alias('total_value'),  # total value per channel
    )
    .orderBy('channel')  # sort by channel name
)
large_txns = (
    silver  # start from clean rows
    .filter(F.col('is_large_txn'))  # keep only large transactions
    .select('transaction_id', 'account_id', 'transaction_date', 'amount', 'channel')  # keep the columns a compliance reviewer needs
)
monthly_account.write.mode('overwrite').parquet(os.path.join(GOLD, 'monthly_account_summary'))  # save the gold table as Parquet
channel_summary.write.mode('overwrite').parquet(os.path.join(GOLD, 'channel_summary'))  # save the channel table as Parquet
large_txns.write.mode('overwrite').parquet(os.path.join(GOLD, 'large_transactions'))  # save the large transactions table as Parquet

## 7. Save CSV extracts
The gold tables are small, so they are converted to pandas and written as CSV files, the same format as the pandas notebook. For very large tables, keep Parquet and do not convert.

In [ ]:
monthly_account.toPandas().to_csv(os.path.join(OUTPUT_DIR, 'monthly_account_summary.csv'), index=False)  # save the monthly account summary as CSV
channel_summary.toPandas().to_csv(os.path.join(OUTPUT_DIR, 'channel_summary.csv'), index=False)  # save the channel summary as CSV
large_txns.toPandas().to_csv(os.path.join(OUTPUT_DIR, 'large_transactions.csv'), index=False)  # save the large transactions list as CSV
silver.toPandas().to_csv(os.path.join(OUTPUT_DIR, 'curated_transactions.csv'), index=False)  # save the clean transactions as CSV
quarantine_df.toPandas().to_csv(os.path.join(OUTPUT_DIR, 'quarantine_transactions.csv'), index=False)  # save the quarantined rows with reason codes as CSV
print('CSV extracts saved to:', OUTPUT_DIR)  # confirm where the files are

In [ ]:
spark.stop()  # release the Spark session and its resources when the job is finished